<a href="https://colab.research.google.com/github/Dubnitskyi/Machine_learning_labs/blob/main/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langchain-community langchain-core sentence-transformers chromadb

In [ ]:
import pandas as pd
import os
import time
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
import chromadb

In [ ]:
# У реальній роботі тут буде завантаження через datasets
data = {
'id': ['doc1', 'doc2', 'doc3', 'doc4', 'doc5'],
'text': [
"Python - високорівнева мова програмування для загального вжитку.",
"Машинне навчання - це підмножина штучного інтелекту.",
"Трансформери змінили підхід до обробки природної мови.",
"ChromaDB - це векторна база даних для зберігання ембедінгів.",
"Метод k-найближчих сусідів використовується для класифікації."
]
}
df = pd.DataFrame(data)


In [ ]:
model_name = 'all-MiniLM-L6-v2'
bi_encoder = SentenceTransformer(model_name)

In [ ]:
client = chromadb.Client()
# Створюємо колекцію.
# Вказуємо функцію ембедінгу, щоб база сама кодувала запити
collection = client.create_collection(name="ms_marco_mini")

In [ ]:
print("Індексація документів...")
collection.add(
documents=df['text'].tolist(),
ids=df['id'].tolist(),
embeddings=bi_encoder.encode(df['text'].tolist()).tolist()
)


In [ ]:
test_queries = [
"Що таке штучний інтелект?",
"Яка мова програмування популярна для ML?",
"Як зберігати вектори?"
]

In [ ]:
bi_encoder_results_table = []

start_time = time.time()

for query in test_queries:
  query_embedding = bi_encoder.encode(query).tolist()
  # Пошук у ChromaDB
  results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10 # Змінено на 10
  )

  # Process results for the current query
  for i in range(len(results['ids'][0])):
    bi_encoder_results_table.append({
      'Query': query,
      'Rank': i + 1,
      'Document ID': results['ids'][0][i],
      'Cosine Score': round(1 - results['distances'][0][i], 4),
      'Text': results['documents'][0][i]
    })

end_time = time.time()
print(f"Час виконання: {end_time - start_time:.4f} секунд")

In [ ]:
results_table = []
for query in test_queries:
  query_embedding = bi_encoder.encode(query).tolist()
  # Пошук у ChromaDB
  results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
  )

In [ ]:
for i in range(len(results['ids'][0])):
  results_table.append({
    'Query': query,
    'Rank': i + 1,
    'Document ID': results['ids'][0][i],
    'Cosine Score': round(1 - results['distances'][0][i], 4),
    'Text': results['documents'][0][i]
  })

In [ ]:
output_df = pd.DataFrame(results_table)
print("\nРезультати пошуку Bi-Encoder:")
print(output_df.to_string(index=False))

# Task 2

In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

In [ ]:
cross_encoder_results_table = []

start_time = time.time()

for query_data in bi_encoder_results_table:
    query = query_data['Query']
    document_id = query_data['Document ID']
    document_text = query_data['Text']

    # Prepare input for cross-encoder
    features = cross_encoder_model.predict([(query, document_text)])

    cross_encoder_results_table.append({
        'Query': query,
        'Document ID': document_id,
        'Text': document_text,
        'Cross-Encoder Score': features.item() # Get the score from the array
    })

end_time = time.time()
print(f"Час виконання Cross-Encoder реранжування: {end_time - start_time:.4f} секунд")

# Group and re-rank for each query
re_ranked_top_3_results = []
for query in test_queries:
    query_specific_results = [res for res in cross_encoder_results_table if res['Query'] == query]
    # Sort by Cross-Encoder Score in descending order
    sorted_results = sorted(query_specific_results, key=lambda x: x['Cross-Encoder Score'], reverse=True)
    # Select Top-3
    for i, result in enumerate(sorted_results[:3]):
        re_ranked_top_3_results.append({
            'Query': result['Query'],
            'Rank (Cross-Encoder)': i + 1,
            'Document ID': result['Document ID'],
            'Cross-Encoder Score': round(result['Cross-Encoder Score'], 4),
            'Text': result['Text']
        })

re_ranked_df = pd.DataFrame(re_ranked_top_3_results)
print("\nРезультати реранжування Top-3 за допомогою Cross-Encoder:")
print(re_ranked_df.to_string(index=False))

In [ ]:
bi_encoder_top_3_results = []
for query in test_queries:
    query_specific_results = [res for res in bi_encoder_results_table if res['Query'] == query]
    # Sort by Cosine Score in descending order for bi-encoder
    sorted_results = sorted(query_specific_results, key=lambda x: x['Cosine Score'], reverse=True)
    # Select Top-3
    for i, result in enumerate(sorted_results[:3]):
        bi_encoder_top_3_results.append({
            'Query': result['Query'],
            'Rank (Bi-Encoder)': i + 1,
            'Document ID': result['Document ID'],
            'Cosine Score': round(result['Cosine Score'], 4),
            'Text': result['Text']
        })

bi_encoder_top_3_df = pd.DataFrame(bi_encoder_top_3_results)
print("\nInitial Top-3 results from Bi-Encoder:")
print(bi_encoder_top_3_df.to_string(index=False))


**Reasoning**:
To compare the rankings effectively, I will merge the `bi_encoder_top_3_df` and `re_ranked_df` DataFrames into a single DataFrame. This will allow for a side-by-side view of the initial and re-ranked results for each query, making it easier to analyze changes.



In [ ]:
merged_df = pd.merge(
    bi_encoder_top_3_df,
    re_ranked_df,
    on=['Query', 'Document ID', 'Text'],
    how='outer'
)

# Fill NaN values that might arise if a document was in one top-3 but not the other
merged_df['Rank (Bi-Encoder)'] = merged_df['Rank (Bi-Encoder)'].fillna('N/A')
merged_df['Cosine Score'] = merged_df['Cosine Score'].fillna('N/A')
merged_df['Rank (Cross-Encoder)'] = merged_df['Rank (Cross-Encoder)'].fillna('N/A')
merged_df['Cross-Encoder Score'] = merged_df['Cross-Encoder Score'].fillna('N/A')

# Reorder columns for better readability
merged_df = merged_df[[
    'Query',
    'Document ID',
    'Text',
    'Rank (Bi-Encoder)',
    'Cosine Score',
    'Rank (Cross-Encoder)',
    'Cross-Encoder Score'
]]

print("\nComparison of Bi-Encoder Top-3 vs. Cross-Encoder Re-ranked Top-3:")
print(merged_df.to_string(index=False))

#Task 3

In [ ]:
import math

def precision_at_k(retrieved_docs_ids, relevant_docs_ids, k):
    if k == 0:
        return 0.0

    hits = 0
    for i in range(min(k, len(retrieved_docs_ids))):
        if retrieved_docs_ids[i] in relevant_docs_ids:
            hits += 1
    return hits / k

def mean_reciprocal_rank(retrieved_docs_ids, relevant_docs_ids):
    for i, doc_id in enumerate(retrieved_docs_ids):
        if doc_id in relevant_docs_ids:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(retrieved_docs_ids, relevance_scores_map, k):
    if k == 0:
        return 0.0

    dcg = 0.0
    for i in range(min(k, len(retrieved_docs_ids))):
        doc_id = retrieved_docs_ids[i]
        # Get relevance, default to 0 if not found (i.e., not considered relevant in ground truth)
        relevance = relevance_scores_map.get(doc_id, 0)
        dcg += relevance / math.log2(i + 2) # i+1 for 1-based rank, +1 for log2 base (log2(1) is 0)

    # Calculate Ideal DCG (IDCG)
    # Sort all known relevant documents by their ideal relevance scores in descending order
    ideal_relevances = sorted(relevance_scores_map.values(), reverse=True)
    idcg = 0.0
    for i in range(min(k, len(ideal_relevances))):
        idcg += ideal_relevances[i] / math.log2(i + 2)

    if idcg == 0:
        return 0.0 # Avoid division by zero if no relevant documents in ideal ranking
    return dcg / idcg

# --- Step 1: Define Ground Truth (Relevant Documents and Graded Relevance) ---
# We'll use the cross-encoder's top 3 for each query as the "ground truth" for relevant documents.
# For graded relevance, we'll assign scores 3, 2, 1 to the top 3 documents from the cross-encoder re-ranking.
# All other documents will have a relevance of 0 for binary metrics and for nDCG if they are not in the top 3 of the cross-encoder results.

ground_truth_relevant_docs = {} # Stores set of relevant doc_ids for each query (binary)
ground_truth_relevance_scores = {} # Stores map of doc_id to graded relevance for each query (nDCG)

for query in test_queries:
    relevant_ids_for_query = set()
    relevance_scores_for_query = {}

    # Get the top 3 re-ranked documents from the cross-encoder for the current query
    query_specific_re_ranked_results = [res for res in re_ranked_top_3_results if res['Query'] == query]

    for i, res in enumerate(query_specific_re_ranked_results):
        doc_id = res['Document ID']
        relevant_ids_for_query.add(doc_id) # For binary relevance

        # Assign graded relevance (3 for rank 1, 2 for rank 2, 1 for rank 3)
        relevance_scores_for_query[doc_id] = 3 - i

    ground_truth_relevant_docs[query] = relevant_ids_for_query
    ground_truth_relevance_scores[query] = relevance_scores_for_query


# --- Step 2: Evaluate Bi-Encoder Results ---
bi_encoder_precision_at_3 = []
bi_encoder_reciprocal_ranks = []
bi_encoder_ndcg_at_3 = []

for query in test_queries:
    # Get the retrieved document IDs from the bi-encoder results (top 10 are available)
    retrieved_by_bi_encoder = [res['Document ID'] for res in bi_encoder_results_table if res['Query'] == query]

    relevant_ids = ground_truth_relevant_docs[query]
    relevance_map = ground_truth_relevance_scores[query]

    # Calculate metrics for bi-encoder
    bi_encoder_precision_at_3.append(precision_at_k(retrieved_by_bi_encoder, relevant_ids, k=3))
    bi_encoder_reciprocal_ranks.append(mean_reciprocal_rank(retrieved_by_bi_encoder, relevant_ids))
    bi_encoder_ndcg_at_3.append(ndcg_at_k(retrieved_by_bi_encoder, relevance_map, k=3))


# --- Step 3: Evaluate Two-Stage (Bi-Encoder + Cross-Encoder) Results ---
two_stage_precision_at_3 = []
two_stage_reciprocal_ranks = []
two_stage_ndcg_at_3 = []

for query in test_queries:
    # Get the retrieved document IDs from the re-ranked results (which are already top 3)
    retrieved_by_two_stage = [res['Document ID'] for res in re_ranked_top_3_results if res['Query'] == query]

    relevant_ids = ground_truth_relevant_docs[query]
    relevance_map = ground_truth_relevance_scores[query]

    # Calculate metrics for two-stage
    two_stage_precision_at_3.append(precision_at_k(retrieved_by_two_stage, relevant_ids, k=3))
    two_stage_reciprocal_ranks.append(mean_reciprocal_rank(retrieved_by_two_stage, relevant_ids))
    two_stage_ndcg_at_3.append(ndcg_at_k(retrieved_by_two_stage, relevance_map, k=3))


# --- Step 4: Aggregate and Print Results ---
num_queries = len(test_queries)

avg_bi_encoder_p_at_3 = sum(bi_encoder_precision_at_3) / num_queries
avg_bi_encoder_mrr = sum(bi_encoder_reciprocal_ranks) / num_queries
avg_bi_encoder_ndcg_at_3 = sum(bi_encoder_ndcg_at_3) / num_queries

avg_two_stage_p_at_3 = sum(two_stage_precision_at_3) / num_queries
avg_two_stage_mrr = sum(two_stage_reciprocal_ranks) / num_queries
avg_two_stage_ndcg_at_3 = sum(two_stage_ndcg_at_3) / num_queries

print("--- Evaluation Metrics ---")
print(f"\nAverage Precision@3:")
print(f"  Bi-Encoder: {avg_bi_encoder_p_at_3:.4f}")
print(f"  Two-Stage (Bi-Encoder + Cross-Encoder): {avg_two_stage_p_at_3:.4f}")

print(f"\nAverage Mean Reciprocal Rank (MRR):")
print(f"  Bi-Encoder: {avg_bi_encoder_mrr:.4f}")
print(f"  Two-Stage (Bi-Encoder + Cross-Encoder): {avg_two_stage_mrr:.4f}")

print(f"\nAverage nDCG@3:")
print(f"  Bi-Encoder: {avg_bi_encoder_ndcg_at_3:.4f}")
print(f"  Two-Stage (Bi-Encoder + Cross-Encoder): {avg_two_stage_ndcg_at_3:.4f}")

#Task 4

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data for plotting
metrics_names = ['Precision@3', 'MRR', 'nDCG@3']
bi_encoder_scores = [avg_bi_encoder_p_at_3, avg_bi_encoder_mrr, avg_bi_encoder_ndcg_at_3]
two_stage_scores = [avg_two_stage_p_at_3, avg_two_stage_mrr, avg_two_stage_ndcg_at_3]

x = np.arange(len(metrics_names)) # the label locations
width = 0.35 # the width of the bars

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, bi_encoder_scores, width, label='Bi-Encoder')
rects2 = ax.bar(x + width/2, two_stage_scores, width, label='Two-Stage (Bi + Cross-Encoder)')

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('Score')
ax.set_title('Comparison of Retrieval Metrics: Bi-Encoder vs. Two-Stage Search')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.set_ylim(0, 1.1) # Set y-axis limit for scores (0 to 1)

def autolabel(rects):
    "Attach a text label above each bar in *rects*, displaying its height."
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom')

autolabel(rects1)
autolabel(rects2)

fig.tight_layout()
plt.show()
